# 과제 3 — 미세조정 · ONNX 배포 · 폐루프 평가
**AI Application · 스마트모빌리티학과 · 배포 2026-10-07 · 마감 10/13(화) 23:59**

`base_pilotnet.pt` 를 여러분의 `my_dataset_v1.npz` 로 미세조정하고, ONNX 로 내보내 폐루프에서 평가합니다. **학습 루프는 `tinytrack.train_bc` 를 쓰지 말고 직접 5단계로** 쓰세요.

| 문제 | 내용 | 배점 |
|---|---|---|
| 1 | 데이터 분할 + 베이스 로드 + 파라미터 수 | 15 |
| 2 | 미세조정 루프 (forward → loss → zero_grad → backward → step) | 15 |
| 3 | 베스트 체크포인트 저장·복원 | 15 |
| 4 | ONNX 내보내기 + torch 일치 검증 (배치 1·8) | 15 |
| 5 | 폐루프 비교표 (expert / base / finetune) | 15 |
| 6 | 실패 프레임 분석 함수 | 15 |
| — | 런타임 → 모두 실행 오류 없음 | 10 |
| 심화 | 동결 vs 전체 미세조정 + 시드 3개 평균 | +10 |

**제출물**: 이 노트북 + `my_policy_v1.onnx` + `compare_v1.csv` + `my_policy_v1_result.json` (npz 제외). 채점 셀 수정 시 0점.

In [1]:
# ============ 0. 준비 (재접속 시 여기부터) ============
import importlib.util, subprocess, sys, os, time, json
for pkg in ["onnx", "onnxruntime", "pandas"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
try:
    import tinytrack as tt
except ImportError:
    from google.colab import files
    print("tinytrack.py 를 업로드하세요"); files.upload(); import tinytrack as tt
try:
    from google.colab import drive
    drive.mount("/content/drive"); ROOT = Path("/content/drive/MyDrive/AIApp")
except Exception:
    ROOT = Path("./outputs")
SAVE_DIR = ROOT / "week06"; SAVE_DIR.mkdir(parents=True, exist_ok=True)
STUDENT = "202212305_김태윤"     # ← 본인 것으로

MODELS = Path("models") if Path("models/base_pilotnet.pt").exists() else Path(".")
if not (MODELS / "base_pilotnet.pt").exists():
    from google.colab import files
    print("base_pilotnet.pt / base_pilotnet.onnx 업로드"); files.upload()
BASE_PT, BASE_ONNX = MODELS / "base_pilotnet.pt", MODELS / "base_pilotnet.onnx"

v1_path, v0_path = ROOT / "week05" / "my_dataset_v1.npz", ROOT / "week04" / "my_dataset_v0.npz"
if v1_path.exists():
    v1 = tt.load_npz(v1_path)
elif v0_path.exists():
    v1 = tt.rebalance(tt.load_npz(v0_path))
else:
    v1 = tt.rebalance(tt.merge(*[tt.collect(tt.ExpertPolicy(), tt.Track.preset(n), 500, seed=10 * ord(n) + k, noise_inject=ni)
                                 for n in "ABC" for k, ni in enumerate([0.0, 0.4, 0.8])]))
torch.manual_seed(0); np.random.seed(0)
PUBLIC = ["A", "B", "C"]
print("tinytrack", tt.__version__, "| 프레임", len(v1["steers"]))

tinytrack 0.3.1 | 프레임 2848


## 문제 1 — 분할 + 베이스 로드 (15점)
`tr, va` 로 run 단위 분할(검증 20%, seed 0)하고, 베이스를 `tt.load_model` 로 불러온 뒤 **학습 가능한 파라미터 수**를 `n_params` 에 계산하세요.

In [2]:
tr, va = tt.split_by_run(v1, 0.2, seed=0)                                   # run 단위 분할 (tinytrack 함수 사용 가능)
base = tt.load_model(BASE_PT)
n_params = sum(p.numel() for p in base.parameters() if p.requires_grad)
print(len(tr["steers"]), len(va["steers"]), n_params)

2257 591 252219


In [3]:
# ---- 채점 1 (수정 금지) ----
assert not (set(np.unique(tr["run_id"])) & set(np.unique(va["run_id"]))), "run 누수"
assert len(tr["steers"]) + len(va["steers"]) == len(v1["steers"]), "프레임 수 불일치"
assert n_params == sum(p.numel() for p in tt.make_model().parameters()), "파라미터 수가 다릅니다"
print("문제 1 통과!")

문제 1 통과!


## 문제 2 — 미세조정 루프 (15점)
`finetune(model, tr, va, epochs, lr)` : 학습 로더는 `tt.make_loader(tr, 64, True, augment=tt.make_augment())`, 검증 로더는 `tt.make_loader(va, 256, False)` 를 쓰고, 옵티마이저는 Adam. 매 에폭 학습 손실과 검증 손실(MSE)을 `hist["train"]`, `hist["val"]` 에 기록해 반환하세요. **5단계 순서**를 지키세요.

In [4]:
def eval_mse(model, loader):
    model.eval(); tot, n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            tot += nn.functional.mse_loss(model(x), y, reduction="sum").item(); n += len(x)
    return tot / n

def finetune(model, tr, va, epochs=3, lr=3e-4):
    train_loader = tt.make_loader(tr, 64, True, augment=tt.make_augment(seed=0))
    val_loader = tt.make_loader(va, 256, False)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    hist = dict(train=[], val=[])
    for ep in range(epochs):
        model.train(); tot, n = 0.0, 0
        for x, y in train_loader:
            pred = model(x)                # ① forward
            loss = crit(pred, y)                # ② loss
            opt.zero_grad()           # ③ zero_grad
            loss.backward()           # ④ backward
            opt.step()                # ⑤ step
            tot += loss.item() * len(x); n += len(x)
        hist["train"].append(tot / n); hist["val"].append(eval_mse(model, val_loader))
        print(f"epoch {ep+1}: train={hist['train'][-1]:.5f} val={hist['val'][-1]:.5f}")
    return hist

val_loader = tt.make_loader(va, 256, False)
base_val = eval_mse(tt.load_model(BASE_PT), val_loader); print("베이스 검증 MSE:", round(base_val, 5))
model = tt.load_model(BASE_PT)
hist = finetune(model, tr, va, epochs=3, lr=3e-4)

베이스 검증 MSE: 0.17622


epoch 1: train=0.06238 val=0.06054


epoch 2: train=0.01832 val=0.03024


epoch 3: train=0.01003 val=0.02971


In [5]:
# ---- 채점 2 (수정 금지) ----
assert len(hist["train"]) == 3 and len(hist["val"]) == 3, "에폭마다 기록해야 합니다"
assert all(np.isfinite(hist["train"])) and all(np.isfinite(hist["val"])), "손실이 발산했습니다 (NaN)"
assert min(hist["val"]) <= base_val * 1.5 + 0.005, "미세조정 후 검증 MSE 가 베이스보다 크게 나빠졌습니다 (학습률·zero_grad 확인)"
assert hist["train"][0] * 0.999 >= hist["train"][-1] or hist["train"][-1] < 0.01, "학습 손실이 내려가지 않습니다"
print("문제 2 통과!")

문제 2 통과!


## 문제 3 — 베스트 체크포인트 (15점)
`finetune_ckpt(model, tr, va, epochs, lr, path)` : 문제 2 와 같되, **검증 MSE 가 최저일 때만** `state_dict` 를 `path` 에 저장하고, 끝나면 그 가중치를 다시 불러온 모델과 `best_val` 을 반환하세요.

In [6]:
def finetune_ckpt(model, tr, va, epochs=4, lr=3e-4, path=SAVE_DIR / "my_policy_v1.pt"):
    train_loader = tt.make_loader(tr, 64, True, augment=tt.make_augment(seed=0))
    val_loader = tt.make_loader(va, 256, False)
    opt = torch.optim.Adam(model.parameters(), lr=lr); crit = nn.MSELoss()
    best_val = float("inf")
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            loss = crit(model(x), y); opt.zero_grad(); loss.backward(); opt.step()
        v = eval_mse(model, val_loader)
        if v < best_val:                                   # 최저 갱신 조건
            best_val = v
            torch.save(model.state_dict(), path)   # state_dict 저장
        print(f"epoch {ep+1}: val={v:.5f} (best {best_val:.5f})")
    model.load_state_dict(torch.load(path))                    # 베스트 복원
    return model, best_val

model, best_val = finetune_ckpt(tt.load_model(BASE_PT), tr, va)

epoch 1: val=0.06235 (best 0.06235)


epoch 2: val=0.03763 (best 0.03763)


epoch 3: val=0.01461 (best 0.01461)


epoch 4: val=0.02025 (best 0.01461)


In [7]:
# ---- 채점 3 (수정 금지) ----
assert (SAVE_DIR / "my_policy_v1.pt").exists(), "체크포인트 파일이 없습니다"
_v = eval_mse(model, tt.make_loader(va, 256, False))
assert abs(_v - best_val) < 1e-6, f"복원된 모델의 검증 MSE({_v:.6f})가 best_val({best_val:.6f})과 다릅니다"
print("문제 3 통과!")

문제 3 통과!


## 문제 4 — ONNX 내보내기 + 일치 검증 (15점)
`tt.export_onnx(model, path)` 로 내보낸 뒤, `parity(model, onnx_path, images)` 가 배치 1 과 8 에 대해 torch 출력과 onnx 출력의 **최대 절대 차이**를 반환하게 하세요. 전처리는 `tt.preprocess` 를 쓰세요.

In [8]:
import onnxruntime as ort
ONNX_PATH = tt.export_onnx(model, SAVE_DIR / "my_policy_v1.onnx")

def parity(model, onnx_path, images):
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    model = model.cpu().eval(); worst = 0.0
    for n in (1, 8):
        x = np.concatenate([tt.preprocess(im) for im in images[:n]])          # 전처리 → (n,3,66,200)
        with torch.no_grad():
            y_t = model(torch.from_numpy(x)).numpy()
        y_o = sess.run(None, {"image": x})[0]                          # 입력 이름
        worst = max(worst, float(np.abs(y_t - y_o).max()))
    return worst

diff = parity(model, ONNX_PATH, va["images"]); print("최대 차이:", diff)

최대 차이: 8.940696716308594e-08


In [9]:
# ---- 채점 4 (수정 금지) ----
assert Path(ONNX_PATH).exists() and os.path.getsize(ONNX_PATH) > 500_000, "ONNX 파일이 없거나 비정상입니다"
assert diff < 1e-4, f"torch 와 onnx 출력이 다릅니다 ({diff:.2e})"
_sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
assert _sess.get_inputs()[0].name == "image" and _sess.get_outputs()[0].name == "steer", "입출력 이름 계약 위반"
print("문제 4 통과!")

문제 4 통과!


## 문제 5 — 폐루프 비교표 (15점)
`compare(policies, tracks)` : `{이름: 정책}` 을 받아 각 정책을 `tt.evaluate(pol, tracks, seed=0)` 로 평가하고, `policy, score, completion, offtrack_rate, mean_abs_d` 열을 가진 DataFrame 을 반환하세요. expert / base / finetune 세 정책으로 공개 트랙에서 실행합니다.

In [10]:
def compare(policies, tracks):
    rows = []
    for name, pol in policies.items():
        r = tt.evaluate(pol, tracks, seed=0)                                            # 평가
        rows.append(dict(policy=name, score=r["score"], completion=r["completion"],
                         offtrack_rate=r["offtrack_rate"], mean_abs_d=r["mean_abs_d"]))
    return pd.DataFrame(rows)

policies = {"expert": tt.ExpertPolicy(), "base": tt.OnnxPolicy(BASE_ONNX), "finetune": tt.OnnxPolicy(ONNX_PATH)}
df = compare(policies, PUBLIC).round(3); print(df.to_string(index=False))
df.to_csv(SAVE_DIR / "compare_v1.csv", index=False)

  policy  score  completion  offtrack_rate  mean_abs_d
  expert 89.824       1.000            0.0       0.102
    base  9.295       0.203            1.0       0.084
finetune 86.691       1.000            0.0       0.133


In [11]:
# ---- 채점 5 (수정 금지) ----
assert list(df["policy"]) == ["expert", "base", "finetune"] and len(df) == 3, "세 정책의 행이 있어야 합니다"
assert {"score", "completion", "offtrack_rate", "mean_abs_d"} <= set(df.columns), "열 이름 확인"
_b, _f = float(df.loc[df.policy == "base", "score"].iloc[0]), float(df.loc[df.policy == "finetune", "score"].iloc[0])
assert _f >= _b - 5, f"미세조정 score({_f:.1f})가 베이스({_b:.1f})보다 5점 이상 낮습니다 — 학습률·체크포인트·전처리 확인"
print("문제 5 통과!")

문제 5 통과!


## 문제 6 — 실패 프레임 분석 (15점)
`analyze_failure(pol, track, seed=0)` : 외란 포함 환경에서 정책을 굴리며 `(d, psi, kappa, steer)` 를 기록하고, 다음 키를 가진 dict 를 반환하세요.
`terminated`(이탈 여부, bool) · `step`(이탈 시점 또는 |d| 최대 시점, int) · `mean_abs_d_last30`(그 시점 직전 30스텝 |d| 평균, float) · `kappa_at_end`(그 시점 곡률, float)

In [12]:
def analyze_failure(pol, track, seed=0, max_steps=1500):
    env = tt.Env(track, disturb=tt.DEFAULT_DISTURB, seed=seed, max_steps=max_steps)
    obs = env.reset(seed=seed); log = []
    for t in range(max_steps):
        a = pol(obs, env.state)
        obs, r, term, trunc, info = env.step(a)
        log.append((info["d"], info["psi"], info["kappa"], a))
        if term or trunc: break
    log = np.array(log)
    end = len(log) - 1 if term else np.argmax(np.abs(log[:, 0]))                     # 이탈이면 마지막, 아니면 |d| 최대 인덱스
    w = log[max(0, end - 30):end + 1]
    return dict(terminated=bool(term), step=int(end), mean_abs_d_last30=float(np.abs(w[:, 0]).mean()), kappa_at_end=float(log[end, 2]))

res = analyze_failure(policies["finetune"], tt.Track.preset("C")); print(res)

{'terminated': False, 'step': 802, 'mean_abs_d_last30': 0.09476915963389543, 'kappa_at_end': 0.25}


In [13]:
# ---- 채점 6 (수정 금지) ----
assert set(res) == {"terminated", "step", "mean_abs_d_last30", "kappa_at_end"}, "키 확인"
assert isinstance(res["terminated"], bool) and isinstance(res["step"], int), "타입 확인"
assert 0 <= res["mean_abs_d_last30"] <= 0.5 + 1e-6 and abs(res["kappa_at_end"]) <= tt.KAPPA_MAX + 1e-6, "값 범위 확인"
_e = analyze_failure(tt.ExpertPolicy(), tt.Track.preset("C"))
assert _e["terminated"] is False, "전문가는 이탈하지 않아야 합니다 (환경 설정 확인)"
print("문제 6 통과!")

문제 6 통과!


## 심화 (선택, +10) — 동결 vs 전체 미세조정, 시드 3개 평균
`freeze_features=True` 로 헤드만 학습한 모델과 전체 미세조정 모델을 시드 0/1/2 로 각각 평가해 평균·표준편차를 표로 만들고, 어느 쪽이 나은지와 이유를 3줄로 쓰세요. (`tt.train_bc` 사용 가능)

In [14]:
m_fr = tt.load_model(BASE_PT)
tt.train_bc(m_fr, tr, va, epochs=3, lr=1e-3, augment=tt.make_augment(seed=0), freeze_features=True, save_path=SAVE_DIR / "freeze_v1.pt", verbose=False)
pol_fr = tt.OnnxPolicy(tt.export_onnx(m_fr, SAVE_DIR / "freeze_v1.onnx"))
rows = []
for name, pol in [("finetune", policies["finetune"]), ("freeze", pol_fr)]:
    sc = [tt.evaluate(pol, PUBLIC, seed=s)["score"] for s in (0, 1, 2)]
    rows.append(dict(policy=name, mean=np.mean(sc), std=np.std(sc), seeds=np.round(sc, 1).tolist()))
print(pd.DataFrame(rows).round(2).to_string(index=False))

  policy  mean   std              seeds
finetune 86.38  0.22 [86.7, 86.2, 86.2]
  freeze 25.95 11.00 [41.5, 18.0, 18.3]


**심화 근거 (3줄):**

1. 시드 0·1·2 평가에서 전체 미세조정 모델의 평균 점수가 동결 모델보다 높아, 이번 조건에서는 전체 미세조정이 더 효과적이었다.
2. 전체 미세조정 모델은 동결 모델보다 점수의 표준편차가 작아, 평가 시드 변화에도 더 안정적인 주행 성능을 보였다.
3. 전체 미세조정은 특징 추출부까지 복구 데이터에 맞게 학습할 수 있어 유리했을 가능성이 있지만, 학습률과 에폭 수도 달라 동결 여부만의 효과로 단정할 수는 없다.

## 결과 저장 · 제출

In [15]:
result = dict(student=STUDENT, data="v1", best_val=best_val, onnx=str(ONNX_PATH), compare=df.to_dict("records"), failure_C=res)
json.dump(result, open(SAVE_DIR / "my_policy_v1_result.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("저장:", SAVE_DIR / "my_policy_v1_result.json")
print("제출물: 이 노트북, my_policy_v1.onnx, compare_v1.csv, my_policy_v1_result.json")

저장: outputs/week06/my_policy_v1_result.json
제출물: 이 노트북, my_policy_v1.onnx, compare_v1.csv, my_policy_v1_result.json


## 제출 방법
1. **런타임 → 모두 실행** 오류 없음 + 채점 셀 6개 `통과!`
2. `과제3_학번_이름.ipynb` 로 저장
3. ① GitHub 개인 저장소 push (노트북 + onnx + csv + json) ② e-Campus ③ 이메일 `[AI응용] 과제3_학번_이름` (저장소 링크). GitHub 이 어려우면 첨부 제출 인정
4. **리더보드**: 같은 `my_policy_v1.onnx` 링크를 Mattermost 에 — 수요일 밤 게시

**마감 10/13(화) 23:59.**